# PedRL — Pedagogical RL on GSM8K (Colab)

Implementation of [Pedagogical RL](https://noahziems.com/pedagogical-rl): train a **privileged teacher** (it sees the gold answer) with GRPO to maximize `correctness × G_spike`, where `G_spike` is a spike-aware learnability score under the **frozen student** — then distill the teacher into the student with **surprisal-gated** cross-entropy.

Repo: [dannyyoon0303/PedRL](https://github.com/dannyyoon0303/PedRL) · Findings so far: [RESULTS.md](https://github.com/dannyyoon0303/PedRL/blob/main/RESULTS.md)

| preset | model / regime | GPU | time |
|---|---|---|---|
| `smoke` | 0.5B, pipeline check | T4 | ~10–15 min |
| `poc` | 0.5B, random GSM8K — **dense rewards** | T4 | ~2–3 h |
| `hard` | Llama-3.2-3B, hard subset — **sparse rewards** (the method's target regime) | **A100** | ~4–5 h |


In [ ]:
!nvidia-smi

## 1. Install dependencies

In [ ]:
%pip install -q -U "trl>=0.17.0" "peft>=0.14.0" "datasets>=2.19.0" "accelerate>=0.34.0" "transformers>=4.49.0"
# Colab preinstalls an old torchao that recent peft chokes on; we don't use it
%pip uninstall -q -y torchao


In [ ]:
# HF token: optional for smoke/poc (public models), REQUIRED for the hard preset
# (Llama-3.2 is gated). Key icon in the left sidebar -> add secret HF_TOKEN.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("no HF_TOKEN configured — fine for smoke/poc; required for --preset hard")

## 2. Get the code

In [ ]:
import os
if os.path.basename(os.getcwd()) == "PedRL":
    !git pull
else:
    if os.path.exists("PedRL"):
        !git -C PedRL pull
    else:
        !git clone https://github.com/dannyyoon0303/PedRL.git
    %cd PedRL
!ls

## 3. Smoke test (~10 min on T4)

Runs the full pipeline on a tiny slice: baseline eval → teacher GRPO → corpus → gated assimilation → student eval. Numbers will be noisy — this only verifies everything runs.

In [ ]:
!python run.py all --preset smoke

## 4. Round 1 — dense-reward PoC (`poc` preset)

256 random GSM8K problems, 0.5B student. **Outcome (see RESULTS.md): the mechanism works — surprisal falls, `G_spike` doubles, correctness holds — but eval doesn't move, and vanilla GRPO wins at matched rollouts, because at 43% pass@1 rewards are dense and blind sampling doesn't need a teacher.** Kept here as the dense-regime contrast for Round 2.

In [ ]:
# fresh outputs for the PoC (the smoke test wrote to outputs/ too)
!rm -rf outputs
!python run.py all --preset poc

## 5. Round-1 results

In [ ]:
# shared plot style + helpers (light surface; series colors: fixed categorical order)
import json, os
import matplotlib.pyplot as plt

C1, C2 = "#2a78d6", "#1baf7a"      # series 1 (blue), series 2 (aqua)
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e0"

def style_ax(ax, xlabel="", ylabel="", title=""):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(GRID)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK2, labelsize=9)
    ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    ax.set_ylabel(ylabel, color=INK2, fontsize=10)
    ax.set_title(title, color=INK, fontsize=11, loc="left", pad=10)

def smooth(xs, w=9):
    if len(xs) < w: return xs
    out = []
    for i in range(len(xs)):
        lo = max(0, i - w // 2); hi = min(len(xs), i + w // 2 + 1)
        out.append(sum(xs[lo:hi]) / (hi - lo))
    return out

def _series(ax, x, y, color, label, dy):
    ax.plot(x, y, color=color, linewidth=1, alpha=0.25)
    ys = smooth(y)
    ax.plot(x, ys, color=color, linewidth=2, label=label)
    ax.annotate(label, (x[-1], ys[-1]), xytext=(6, dy),
                textcoords="offset points", color=INK2, fontsize=9, va="center")

def plot_teacher_log(log_path):
    """Surprisal / learnability over teacher training, from reward_log_*.jsonl."""
    log = [json.loads(l) for l in open(log_path)]
    x = [r["rollouts"] for r in log]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    fig.patch.set_facecolor("white")

    _series(ax1, x, [r["mean_gap"] for r in log], C1, "mean gap", 6)
    _series(ax1, x, [r["mean_max_gap"] for r in log], C2, "max gap", -6)
    style_ax(ax1, "teacher rollouts", "surprisal gap d_t (nats)",
             "Student surprisal of teacher completions")
    ax1.legend(frameon=False, fontsize=9, labelcolor=INK2)
    ax1.margins(x=0.14)

    _series(ax2, x, [r["mean_g"] for r in log], C1, "G_spike", 6)
    _series(ax2, x, [r["acc"] for r in log], C2, "teacher acc", -6)
    style_ax(ax2, "teacher rollouts", "score (0-1)", "Learnability and correctness")
    ax2.set_ylim(0, 1.02)
    ax2.legend(frameon=False, fontsize=9, labelcolor=INK2)
    ax2.margins(x=0.14)
    plt.tight_layout(); plt.show()

def plot_curves(out_dir):
    """Sample efficiency: pass@1 vs rollouts for PedRL vs vanilla GRPO."""
    fig, ax = plt.subplots(figsize=(7, 4.2))
    fig.patch.set_facecolor("white")

    base_file = os.path.join(out_dir, "eval_base_hard.json")
    if not os.path.exists(base_file):
        base_file = os.path.join(out_dir, "eval_base.json")
    with open(base_file) as f:
        base_acc = json.load(f)["accuracy"]
    ax.axhline(base_acc, color=INK2, linewidth=1.2, linestyle=(0, (4, 4)))
    ax.annotate("base model", (0, base_acc), xytext=(4, 5),
                textcoords="offset points", color=INK2, fontsize=9)

    for i, (name, color, label) in enumerate([
            ("pedrl", C1, "PedRL (teacher rollouts)"),
            ("baseline", C2, "vanilla GRPO")]):
        with open(os.path.join(out_dir, f"curve_{name}.json")) as f:
            pts = json.load(f)["points"]
        xs = [p["rollouts"] for p in pts]
        ys = [p["accuracy"] for p in pts]
        ax.plot(xs, ys, color=color, linewidth=2, marker="o", markersize=6, label=label)
        ax.annotate(label, (xs[-1], ys[-1]), xytext=(8, 6 if i == 0 else -6),
                    textcoords="offset points", color=INK2, fontsize=9, va="center")

    style_ax(ax, "training rollouts", "GSM8K pass@1",
             "Sample efficiency: PedRL vs vanilla GRPO")
    ax.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="lower right")
    ax.margins(x=0.2)
    ax.set_ylim(bottom=0)
    plt.tight_layout(); plt.show()

def print_evals(out_dir):
    import glob
    for path in sorted(glob.glob(os.path.join(out_dir, "eval_*.json"))):
        with open(path) as f:
            r = json.load(f)
        print(f"{r['tag']:>26}: pass@1 = {r['accuracy']:.3f}  (n={r['n']})")

def plot_probe(out_dir):
    """Premise check: reward density of blind vs privileged sampling on hard problems."""
    with open(os.path.join(out_dir, "probe.json")) as f:
        p = json.load(f)
    metrics = [("pass1", "pass@1\n(per-sample)"),
               ("any_correct", ">=1 correct\nin group"),
               ("mixed_groups", "mixed groups\n(GRPO-learnable)")]
    series = [("privileged", C1, "privileged (hint, untrained)"),
              ("blind", C2, "blind (student)")]
    fig, ax = plt.subplots(figsize=(7, 3.8))
    fig.patch.set_facecolor("white")
    w, xs = 0.34, range(len(metrics))
    for si, (cond, color, label) in enumerate(series):
        vals = [p["conditions"][cond][m] for m, _ in metrics]
        pos = [x + (si - 0.5) * (w + 0.04) for x in xs]
        ax.bar(pos, vals, width=w, color=color, label=label, zorder=2)
        for x, v in zip(pos, vals):
            ax.annotate(f"{v:.2f}", (x, v), xytext=(0, 4), textcoords="offset points",
                        ha="center", color=INK2, fontsize=9)
    ax.set_xticks(list(xs))
    ax.set_xticklabels([lbl for _, lbl in metrics], color=INK2, fontsize=9)
    style_ax(ax, "", "fraction",
             f"Reward density on {p['n']} hard problems ({p['k']} samples each)")
    ax.set_ylim(0, 1.05)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK2)
    plt.tight_layout(); plt.show()


In [ ]:
print_evals("outputs")

### 5a. Does surprisal decrease as the teacher trains?

If the pedagogical objective works, the surprisal gaps `d_t` (under the frozen student) fall and `G_spike` rises while correctness stays high. Note the near-zero flat `mean gap`: privileged off-policyness lives in a few spike tokens at reasoning forks — the empirical case for the *spike-aware* score.

In [ ]:
plot_teacher_log("outputs/reward_log_teacher.jsonl")

In [ ]:
# peek at a teacher demonstration the student found most learnable
import json
rows = [json.loads(l) for l in open("outputs/distill_corpus.jsonl")]
rows.sort(key=lambda r: -r["g_spike"])
r = rows[0]
print(f"G_spike={r['g_spike']:.3f}  gold={r['answer']}\n")
print(r["completion_text"][:1500])

### 5b. Sample efficiency vs vanilla GRPO

Matched rollout budgets; PedRL's step-0 point = distillation from the *untrained* privileged teacher (plain rejection sampling), isolating what pedagogical training adds.

In [ ]:
# vanilla-GRPO baseline at the same rollout budget as the teacher (writes checkpoints too)
!python run.py baseline-rl --preset poc

In [ ]:
# learning curves: eval accuracy vs rollouts (writes outputs/curve_*.json)
!python run.py curve-baseline --preset poc
!python run.py curve-pedrl --preset poc

In [ ]:
plot_curves("outputs")

## 6. Round 2 — the hard subset on A100 (`hard` preset)

**The experiment that tests the method's actual claim.** Round 1 showed privileged teaching has no edge when rewards are dense. The `hard` preset reconstructs the blog's sparse regime:

- `filter-hard` keeps only problems the base student fails **0/4** samples → on-policy RL reward ≈ 0 by construction (vanilla GRPO should **stall** here), and every correct teacher demo carries information the student lacks.
- Model: **Llama-3.2-3B-Instruct** — the blog's model, chosen there explicitly because Qwen is math-mid-trained. ⚠️ Gated: accept the license at [huggingface.co/meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) and set the `HF_TOKEN` secret (cell in section 1). Ungated fallback: append `--set model_name=Qwen/Qwen2.5-1.5B-Instruct` to every command.
- Hint prompt is now direct ("here is the answer, use it as a hint, land on it"); stealth is enforced by the surprisal reward, not the prompt.
- Every eval also writes a held-out **hard slice** (`eval_*_hard.json`) — where the effect must appear first. All artifacts go to `outputs_hard/`.

⏱️ A100: `all` ≈ 3.5–4 h (filter-hard ~1–1.5 h of that); baseline + curves ≈ 2–3 h more. Stages are resumable — `filter-hard` skips if its files exist.

### 6a. Premise check (run BEFORE the 4-hour run)

The method's key motivation: on hard problems a **blind sampler almost never finds correct rollouts** (GRPO groups have zero reward variance → zero gradient → vanilla RL stalls), while the **privileged sampler does** — even before any training. `probe-hard` measures exactly this: the same base model, k samples per problem, with and without the hint (~30 min on A100).

If the premise holds (blind ≈ 0, privileged substantially higher), the 4-hour run is worth it. If privileged is *also* ≈ 0, the hint isn't giving the 3B model enough kick — switch to `--set privileged=solution` before training.

In [ ]:
!python run.py filter-hard --preset hard
!python run.py probe-hard --preset hard

In [ ]:
plot_probe("outputs_hard")

In [ ]:
# full round-2 pipeline (filter-hard already done above, it will be skipped):
# eval-base -> teacher -> corpus -> assimilate -> eval-student
!python run.py all --preset hard

In [ ]:
print_evals("outputs_hard")

In [ ]:
plot_teacher_log("outputs_hard/reward_log_teacher.jsonl")

In [ ]:
# the money plot: on the hard slice, vanilla GRPO should stall while PedRL climbs
!python run.py baseline-rl --preset hard
!python run.py curve-baseline --preset hard
!python run.py curve-pedrl --preset hard

In [ ]:
plot_curves("outputs_hard")

## 7. Optional: ablations

- **No-gating** — plain rejection-sampling SFT on the same corpus: is the surprisal gate doing work?
- **`privileged=solution`** — hand the teacher the full reference solution (the small-model fallback).
- **Stage 3** — continue with standard GRPO on the assimilated student.

In [ ]:
# ablation: plain SFT on the same corpus (writes eval_student_sft_ablation.json)
!python run.py assimilate --preset hard --no-gating
!python run.py eval-student --preset hard --no-gating

In [ ]:
# optional stage 3: standard GRPO on the student (writes eval_student_rl.json)
!python run.py student-rl --preset hard

---
References: [Pedagogical RL blog](https://noahziems.com/pedagogical-rl) · [OPSD](https://github.com/siyan-zhao/OPSD) · [GRPO](https://arxiv.org/abs/2402.03300)